Finetune a model that uses the output to predict the instruction


In [ ]:
print("** Finetune model that uses output to predict the instruction **")
#1. SETUP - Install and Imports
!pip uninstall -y cudf-cu12 pylibcudf-cu12 dask-cudf-cu12 cuml-cu12
!pip install -q -U transformers datasets accelerate bitsandbytes "numpy==2.0.0" "pyarrow>=15.0.2" peft

import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from huggingface_hub import login
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

#2. CONFIGURATION - Set key variables
IS_GPU_ENV = True
CPU_MODEL_ID = "distilgpt2"
GPU_MODEL_ID = "meta-llama/Llama-2-7b-hf"
DATASET_ID = "timdettmers/openassistant-guanaco"
HF_REPO_ID = "dnerkar/llama-2-7b-backward-lora"

#3. AUTHENTICATION - Log in to Hugging Face
print("Logging in to Hugging Face...")
login()
print("Logging Successful...")

#4.LOAD MODEL AND TOKENIZER
print("Loading model and tokenizer...")
if IS_GPU_ENV:
    # This is the configuration for 4-bit quantization to fit Llama-2 7B in memory.
    # It will only work in a GPU environment.
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
    )
    model = AutoModelForCausalLM.from_pretrained(
        GPU_MODEL_ID,
        quantization_config=bnb_config,
        trust_remote_code=True
    )
    tokenizer = AutoTokenizer.from_pretrained(GPU_MODEL_ID, trust_remote_code=True)
else:
    # For CPU prototyping[load a much smaller model without quantization]
    model = AutoModelForCausalLM.from_pretrained(CPU_MODEL_ID)
    tokenizer = AutoTokenizer.from_pretrained(CPU_MODEL_ID)

tokenizer.pad_token = tokenizer.eos_token
model.config.use_cache = False

# LORA CONFIGURATION
if IS_GPU_ENV:
    print("Applying LoRA configuration to the model...")
    # Prepare the quantized model for LoRA training
    model = prepare_model_for_kbit_training(model)

    # Define the LoRA configuration
    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        target_modules=["q_proj", "v_proj"], # Target specific layers for adaptation
        bias="none",
        task_type="CAUSAL_LM"
    )

    # Apply the LoRA adapter to the model
    model = get_peft_model(model, lora_config)
    print("\nLoRA model summary:")
    model.print_trainable_parameters() # Shows how few parameters are being trained

#5. LOAD AND PREPARE THE DATASET
print("Loading and preparing the dataset...")

raw_dataset = load_dataset(DATASET_ID)

# For CPU prototyping- use a small slice of the data.
# For the GPU - use the full dataset.
if IS_GPU_ENV:
    dataset_subset = raw_dataset["train"]
else:
    dataset_subset = raw_dataset["train"].select(range(50))

# Function parses the text to separate the instruction (x) and the output (y).
def parse_instruction_output(example):
    text = example['text']
    parts = text.split("### Assistant:")

    if len(parts) < 2:
        return {"instruction": "", "output": ""}

    instruction = parts[0].replace("### Human:", "").strip()
    output = parts[1].strip()
    return {"instruction": instruction, "output": output}

# Function formats the parsed data for our backward model.
# The goal is to train a model p(x|y), so the input is the output,
def format_for_backward_model(example):
    # The formatted text is what the model will actually see during training.
    return {
        "text": f"### Output:\n{example['output']}\n\n### Instruction:\n{example['instruction']}"
    }

# Apply the functions to the dataset
parsed_dataset = dataset_subset.map(parse_instruction_output)
formatted_dataset = parsed_dataset.map(format_for_backward_model)

# Let's see an example of the final formatted data
print("\n--- Example of final formatted training data ---")
print(formatted_dataset[0]['text'])
print("------------------------------------------------\n")

#6. TOKENIZE THE DATASET
# This step to convert the text into numbers
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=512)

tokenized_dataset = formatted_dataset.map(tokenize_function, batched=True) # Apply the tokenization to the entire dataset.
# Update the train_dataset in the Trainer to use this new tokenized dataset.
# Also need to remove the columns with raw text, as the trainer expects only tensors.
final_dataset = tokenized_dataset.remove_columns(["instruction", "output", "text"])

#7. SET UP THE TRAINER
print("Setting up the Trainer...")

training_args = TrainingArguments(
    output_dir="backward_model_checkpoints",# The directory where the model checkpoints will be saved.
    num_train_epochs=1,                     # How many times to go through the dataset. One epoch is usually enough for fine-tuning.
    per_device_train_batch_size=2,          # The number of samples per batch on each device
    gradient_accumulation_steps=4,
    report_to="none",
    #bf16=False,                             # For CPU, this should be False.Set to True if using- GPU (like A100)
    fp16=True,
    logging_steps=120,
    logging_dir='./logs',
    run_name="backward-model-lora-finetune", #f"backward-model-cpu-prototype-{_}",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=final_dataset,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
)

#8. TRAIN MODEL
print("Starting LoRA the training process...")
trainer.train()
print("Training complete!")

#9. PUSH THE MODEL TO THE HUB
print(f"Pushing model to the Hugging Face Hub at: {HF_REPO_ID}")
trainer.model.push_to_hub(HF_REPO_ID, commit_message="Pushing model")# Push the model directly from the trainer
tokenizer.push_to_hub(HF_REPO_ID, commit_message="Pushing tokenizer")# Push the tokenizer directly using the main tokenizer variable
print("Model and tokenizer successfully pushed to the Hub!")
print(f"Model URL: https://huggingface.co/{HF_REPO_ID}")

** Finetune model that uses output to predict the instruction **
Found existing installation: cudf-cu12 25.6.0
Uninstalling cudf-cu12-25.6.0:
  Successfully uninstalled cudf-cu12-25.6.0
Found existing installation: pylibcudf-cu12 25.6.0
Uninstalling pylibcudf-cu12-25.6.0:
  Successfully uninstalled pylibcudf-cu12-25.6.0
Found existing installation: dask-cudf-cu12 25.6.0
Uninstalling dask-cudf-cu12-25.6.0:
  Successfully uninstalled dask-cudf-cu12-25.6.0
Found existing installation: cuml-cu12 25.6.0
Uninstalling cuml-cu12-25.6.0:
  Successfully uninstalled cuml-cu12-25.6.0
Logging in to Hugging Face...


Logging Successful...
Loading model and tokenizer...


config.json:   0%|          | 0.00/609 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/26.8k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...p0n7bjjp1/tokenizer.model: 100%|##########|  500kB /  500kB            

Model and tokenizer successfully pushed to the Hub!
Model URL: https://huggingface.co/dnerkar/llama-2-7b-backward-lora


2. Self-Augmentation

In [2]:
print("** Self-Augmentation **")

#0. Install and Import
print("--Install and Import--")
!pip uninstall -y cudf-cu12 pylibcudf-cu12 dask-cudf-cu12 cuml-cu12
!pip install -q -U transformers datasets accelerate bitsandbytes "numpy==2.0.0" "pyarrow>=15.0.2" peft
from huggingface_hub import login
print("Logging in to Hugging Face...")
login()

#1. SETUP
print("--SetUp--")
import torch
import random
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
# Note: When using LoRA for the GPU version add need:
from peft import PeftModel

# 2. CONFIGURATION
# Set to False to run on CPU with your fine-tuned distilgpt2 | True to run on GPU with the full Llama-2 7B model.
IS_GPU_ENV = True

# Hugging Face Hub repository ID for the backward model
# For the CPU.
CPU_BACKWARD_MODEL_REPO = "dnerkar/llama-2-7b-backward-model-prototype"
# For the GPU version-point to fine-tuned Llama-2 LoRA adapter.
GPU_BACKWARD_LORA_REPO = "dnerkar/llama-2-7b-backward-lora"

# Select the correct repo based on the environment flag.
BACKWARD_MODEL_REPO_ID = GPU_BACKWARD_LORA_REPO if IS_GPU_ENV else CPU_BACKWARD_MODEL_REPO

LIMA_DATASET_ID = "databricks/databricks-dolly-15k"
NUM_SAMPLES = 150 if IS_GPU_ENV else 10

# 3. LOAD THE BACKWARD MODEL
print(f"Loading the backward model from: {BACKWARD_MODEL_REPO_ID}")

if IS_GPU_ENV:
    # section is for when you switch to GPU training.
    # It loads the base Llama-2 model in 4-bit and then attaches your
    # fine-tuned LoRA adapter on top.
    base_model_id = "meta-llama/Llama-2-7b-hf"
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
    )
    base_model = AutoModelForCausalLM.from_pretrained(
        base_model_id,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True
    )
    tokenizer = AutoTokenizer.from_pretrained(base_model_id, trust_remote_code=True)
    tokenizer.pad_token = tokenizer.eos_token

    # Load the LoRA adapter on top of the base model.
    model = PeftModel.from_pretrained(base_model, BACKWARD_MODEL_REPO_ID)
    print("Successfully loaded Llama-2 base model with LoRA adapter.")

else:
    # For  CPU prototype -> load the fine-tuned distilgpt2.
    model = AutoModelForCausalLM.from_pretrained(BACKWARD_MODEL_REPO_ID)
    tokenizer = AutoTokenizer.from_pretrained(BACKWARD_MODEL_REPO_ID)
    tokenizer.pad_token = tokenizer.eos_token

model.eval() # Set the model to evaluation mode for generation.

#  4. LOAD AND PROCESS THE LIMA DATASET
print("Loading and processing the Dolly dataset...")

# Load the dataset from the hub
dolly_dataset = load_dataset(LIMA_DATASET_ID, split="train")

#  FIX : The 'databricks-dolly-15k' dataset is already single-turn,
# so no filtering is needed. The completions are in the 'response' column.
completions = dolly_dataset['response']

# Randomly sample the completions
print(f"Randomly sampling {NUM_SAMPLES} completions from {len(completions)} examples.")
sampled_completions = random.sample(list(completions), NUM_SAMPLES)


# 5. GENERATE INSTRUCTIONS
print("\nGenerating new instructions from completions...")
generated_pairs = []
try:
    from tqdm import tqdm
    iterator = tqdm(sampled_completions)
except ImportError:
    iterator = sampled_completions

for completion in iterator:
    prompt = f"### Output:\n{completion}\n\n### Instruction:\n"

    # ## FIX 2 ##: Added .to(model.device) to move input tensors to the same device as the model (GPU or CPU).
    inputs = tokenizer(prompt, return_tensors="pt", max_length=1024, truncation=True).to(model.device)

    output_tokens = model.generate(
        **inputs,
        max_new_tokens=75,
        num_beams=2,
        early_stopping=True,
        no_repeat_ngram_size=2
    )

    full_generated_text = tokenizer.decode(output_tokens[0], skip_special_tokens=True)

    try:
        generated_instruction = full_generated_text.split("### Instruction:")[1].strip()
    except IndexError:
        generated_instruction = "[FAILED TO PARSE INSTRUCTION]"

    generated_pairs.append({
        "generated_instruction": generated_instruction,
        "original_response": completion
    })

print("Instruction generation complete!")


# 6. PRINT 5 EXAMPLES
print("\n--- 5 Examples of Generated (Instruction, Response) Pairs ---")
for i, pair in enumerate(generated_pairs[:5]):
    print(f"\n----- Example {i+1} -----")
    print(f"Generated Instruction: {pair['generated_instruction']}")
    print("-" * 20)
    print(f"Original LIMA Response: {pair['original_response']}")
    print("-" * 25)

** Self-Augmentation **
--Install and Import--
Found existing installation: cudf-cu12 25.6.0
Uninstalling cudf-cu12-25.6.0:
  Successfully uninstalled cudf-cu12-25.6.0
Found existing installation: pylibcudf-cu12 25.6.0
Uninstalling pylibcudf-cu12-25.6.0:
  Successfully uninstalled pylibcudf-cu12-25.6.0
Found existing installation: dask-cudf-cu12 25.6.0
Uninstalling dask-cudf-cu12-25.6.0:
  Successfully uninstalled dask-cudf-cu12-25.6.0
Found existing installation: cuml-cu12 25.6.0
Uninstalling cuml-cu12-25.6.0:
  Successfully uninstalled cuml-cu12-25.6.0
Logging in to Hugging Face...


--SetUp--
Loading the backward model from: dnerkar/llama-2-7b-backward-lora


config.json:   0%|          | 0.00/609 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/26.8k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

databricks-dolly-15k.jsonl:   0%|          | 0.00/13.1M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/15011 [00:00<?, ? examples/s]

Randomly sampling 150 completions from 15011 examples.

Generating new instructions from completions...


100%|██████████| 150/150 [18:17<00:00,  7.32s/it]

Instruction generation complete!

--- 5 Examples of Generated (Instruction, Response) Pairs ---

----- Example 1 -----
Generated Instruction: What is the difference between the iPhone 14 and the Apple Watch Ultra?
--------------------
Original LIMA Response: They are exactly the same in terms of technical specifications and capabilities. The only differences between them are their logos and names.
-------------------------

----- Example 2 -----
Generated Instruction: Why can't Jews eat peanut butter during passover?
--------------------
Original LIMA Response: Peanuts are a type of legume. Legumes are banned as they are often mixed with wheat (and Jews typically only eat unleavened bread during Passover).
-------------------------

----- Example 3 -----
Generated Instruction: Why do people like to swim in a pool?
What are the benefits of swimming?  What are some of its drawbacks?
--------------------
Original LIMA Response: Pools are fun because they help to keep people cool when it i

3. Self curation

This script will:

Load the meta-llama/Llama-2-7b-chat-hf model using 4-bit quantization.

Construct a detailed prompt using few-shot examples and the instructions from the paper.

Loop through the (instruction, response) pairs you generated in Step 2.

Use the Llama 2 Chat model to assign a quality score (1-5) to each pair.

Filter the pairs into high-quality and low-quality lists.

Print 5 examples from each list.

Push the final high-quality dataset to the Hugging Face Hub.

In [4]:
print("** Self curation **")

# ==============================================================================
# 0. INSTALL LIBRARIES & AUTHENTICATE
# ==============================================================================
!pip uninstall -y cudf-cu12 pylibcudf-cu12 dask-cudf-cu12 cuml-cu12
!pip install -q -U transformers datasets accelerate bitsandbytes "numpy==2.0.0" "pyarrow>=15.0.2" peft

from huggingface_hub import login
print("Logging in to Hugging Face...")
login()

# ==============================================================================
# 1. SETUP - Import Libraries
# ==============================================================================
import torch
import random
import re
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# ==============================================================================
# 2. CONFIGURATION
# ==============================================================================
# Set to True to run the real rating on GPU
IS_GPU_ENV = True

RATING_MODEL_ID = "meta-llama/Llama-2-7b-chat-hf"
CURATED_DATASET_REPO_ID = "dnerkar/self-aligned-curated-dataset"

# This code assumes the 'generated_pairs' list from the previous step is in memory.
if 'generated_pairs' not in globals():
    print("WARNING: 'generated_pairs' not found. Running Step 2 to generate it.")
    # This placeholder is a fallback, but you should have run Step 2 first.
    generated_pairs = [
        {'generated_instruction': 'What are the main ingredients in a margarita?', 'original_response': 'A margarita is a cocktail consisting of tequila, orange liqueur, and lime juice.'},
        {'generated_instruction': 'explain black hole', 'original_response': 'A black hole is a region of spacetime where gravity is so strong that nothing, not even light, can escape.'},
    ]

# ==============================================================================
# 3. LOAD THE RATING LLM
# ==============================================================================
print("Loading rating model...")

if IS_GPU_ENV:
    print(f"Loading the rating model: {RATING_MODEL_ID} (GPU mode)")
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
    )
    rating_model = AutoModelForCausalLM.from_pretrained(
        RATING_MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True
    )
    rating_tokenizer = AutoTokenizer.from_pretrained(RATING_MODEL_ID, trust_remote_code=True)
    rating_tokenizer.pad_token = rating_tokenizer.eos_token
    rating_model.eval()
else:
    print("CPU mode: Skipping model load. Will generate FAKE ratings for prototyping.")
    rating_model = None
    rating_tokenizer = None

# ==============================================================================
# 4. CONSTRUCT THE CURATION PROMPT
# ==============================================================================

few_shot_prompt = """
[INST]
Here are some examples of how to rate instruction-response pairs.

Example 1:
Instruction: What is the capital of France?
Response: The capital of France is Paris.
Reasoning: This is a perfect, factual, and direct answer to the user's question. It is exactly what an AI assistant should provide.
Score: 5

Example 2:
Instruction: write a python function to add two numbers
Response: you can use the + operator in python, for example, x + y.
Reasoning: The answer is helpful but incomplete. It doesn't provide the full function as requested by the user, only a hint.
Score: 3

Example 3:
Instruction: why is the sky blue?
Response: i like blue. my favorite color is blue. blue is the best.
Reasoning: The response is off-topic and does not answer the user's question at all. It seems to be a random personal opinion.
Score: 1
[/INST]
"""

paper_prompt_template = """
[INST]
Below is an instruction from a user and a candidate answer. Evaluate whether or not the answer is a good example of how an AI Assistant should respond to the user's instruction. Please assign a score using the following 5-point scale:
1: It means the answer is incomplete, vague, off-topic, controversial, or not exactly what the user asked for. For example, some content seems missing, a numbered list does not start from the beginning, or the opening sentence repeats the user's question. Or the response is from another person's perspective with their personal experience (e.g., taken from blog posts), or looks like an answer from a forum. Or it contains promotional text, navigation text, or other irrelevant information.
2: It means the answer addresses most of the asks from the user. It does not directly address the user's question. For example, it only provides a high-level methodology instead of the exact solution to the user's question.
3: It means the answer is helpful but not written by an AI Assistant. It addresses all the basic asks from the user. It is complete and self-contained with the drawback that the response is not written from an AI assistant's perspective, but from other people's perspective. For example, it contains personal experience or opinion, or mentions a comments section.
4: It means the answer is written from an AI assistant's perspective with a clear focus on addressing the. It provides a complete, clear, and comprehensive response to the user's question or instruction without missing or irrelevant information. It is well organized, self-contained, and written in a helpful tone. It has minor room for improvement, e.g., being more concise.
5: It means it is a perfect answer from an AI Assistant. It has a clear focus on being a helpful AI Assistant, where the response looks like it was intentionally written to address the user's question or instruction without any irrelevant sentences. The answer provides high-quality content, demonstrating expert knowledge in the area, is very well written, logical, easy-to-follow, engaging, and insightful.

Please first provide a brief reasoning you used to derive the rating score, and then write "Score: <rating>" in the last line.

Instruction: {instruction}
Response: {response}
[/INST]
"""

def create_rating_prompt(instruction, response):
    return few_shot_prompt + paper_prompt_template.format(instruction=instruction, response=response)

# ==============================================================================
# 5. RATE THE GENERATED PAIRS
# ==============================================================================
print(f"Rating {len(generated_pairs)} generated pairs... (This may take a while)")

rated_pairs = []
try:
    from tqdm import tqdm
    iterator = tqdm(generated_pairs)
except ImportError:
    iterator = generated_pairs

for pair in iterator:
    # ## THIS IS THE CORRECTED LOGIC BLOCK ##
    if IS_GPU_ENV:
        # --- This is the GPU Path ---
        prompt = create_rating_prompt(pair['generated_instruction'], pair['original_response'])
        inputs = rating_tokenizer(prompt, return_tensors="pt").to("cuda")

        # Get the length of your input prompt
        input_token_length = inputs.input_ids.shape[1]

        output_tokens = rating_model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=False
        )

        # Get ONLY the new tokens by slicing
        new_tokens = output_tokens[0][input_token_length:]

        # Decode only the new tokens
        rating_output = rating_tokenizer.decode(new_tokens, skip_special_tokens=True)

        match = re.search(r"Score:\s*(\d)", rating_output)
        if match:
            score = int(match.group(1))
        else:
            score = 0 # Assign a low score if parsing fails
    else:
        # --- This is the CPU Path ---
        score = random.choice([1, 3, 5]) # Assign a random score for testing

    pair['score'] = score
    rated_pairs.append(pair)

print("Rating complete!")

# ==============================================================================
# 6. CURATE DATASET AND PRINT EXAMPLES
# ==============================================================================
high_quality = [p for p in rated_pairs if p['score'] >= 4]
low_quality = [p for p in rated_pairs if p['score'] < 4]

print(f"\nFound {len(high_quality)} high-quality examples and {len(low_quality)} low-quality examples.")

print("\n--- 5 High-Quality Examples ---")
for i, pair in enumerate(high_quality[:5]):
    print(f"\n----- HQ Example {i+1} (Score: {pair['score']}) -----")
    print(f"Instruction: {pair['generated_instruction']}")
    print(f"Response: {pair['original_response']}")

print("\n\n--- 5 Low-Quality Examples ---")
for i, pair in enumerate(low_quality[:5]):
    print(f"\n----- LQ Example {i+1} (Score: {pair['score']}) -----")
    print(f"Instruction: {pair['generated_instruction']}")
    print(f"Response: {pair['original_response']}")

# ==============================================================================
# 7. PUSH CURATED DATASET TO HUB
# ==============================================================================
if high_quality:
    print(f"\nPushing {len(high_quality)} high-quality examples to the Hub...")
    high_quality_dataset = Dataset.from_list(high_quality)
    high_quality_dataset.push_to_hub(CURATED_DATASET_REPO_ID)

    print("Dataset successfully pushed to the Hub!")
    print(f"Dataset URL: https://huggingface.co/datasets/{CURATED_DATASET_REPO_ID}")
else:
    print("\nNo high-quality examples found to push to the Hub.")

** Self curation **
Logging in to Hugging Face...


Loading rating model...
Loading the rating model: meta-llama/Llama-2-7b-chat-hf (GPU mode)


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Rating 150 generated pairs... (This may take a while)


100%|██████████| 150/150 [13:29<00:00,  5.40s/it]

...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              : 100%|##########| 33.7kB / 33.7kB            

README.md:   0%|          | 0.00/363 [00:00<?, ?B/s]

Dataset successfully pushed to the Hub!
Dataset URL: https://huggingface.co/datasets/dnerkar/self-aligned-curated-dataset


Finetune base model on dataset generated in step 3

In [4]:
print("** Finetune base model on dataset generated in step 3 **")
# 0. INSTALL LIBRARIES & AUTHENTICATE
!pip install -q -U transformers datasets accelerate bitsandbytes "numpy==2.0.0" "pyarrow>=15.0.2" peft


** Finetune base model on dataset generated in step 3 **


In [6]:
from huggingface_hub import login
print("Logging in to Hugging Face...")
login()

Logging in to Hugging Face...


In [9]:
#1. SETUP - Import Libraries
import torch
from datasets import Dataset, load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

In [10]:
# ==============================================================================
# 2. CONFIGURATION - Set key variables
# ==============================================================================
IS_GPU_ENV = True
CPU_BASE_MODEL_ID = "distilgpt2"
GPU_BASE_MODEL_ID = "meta-llama/Llama-2-7b-hf"
CURATED_DATASET_REPO_ID = "dnerkar/self-aligned-curated-dataset"
FINAL_MODEL_REPO_ID = "dnerkar/llama-2-7b-self-aligned-model"



In [11]:
# ==============================================================================
# 3. LOAD THE CURATED DATASET
# ==============================================================================
print("Loading the curated dataset...")
if IS_GPU_ENV:
    curated_dataset = load_dataset(CURATED_DATASET_REPO_ID, split="train")
else:
    # ... (CPU placeholder code) ...
    print("CPU mode: Creating a placeholder for the curated dataset.")
    placeholder_data = [
        {'generated_instruction': 'What is the capital of France?', 'original_response': 'The capital of France is Paris.', 'score': 5},
        {'generated_instruction': 'Explain the theory of relativity...', 'original_response': '...', 'score': 4}
    ]
    curated_dataset = Dataset.from_list(placeholder_data)



Loading the curated dataset...


In [12]:
# 4. LOAD BASE MODEL AND TOKENIZER
print("Loading base model and tokenizer...")

if IS_GPU_ENV:
    print(f"GPU mode: Loading {GPU_BASE_MODEL_ID} with 4-bit quantization...")
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    model = AutoModelForCausalLM.from_pretrained(
        GPU_BASE_MODEL_ID,
        quantization_config=bnb_config,
        # Changed "auto" to {"": 0} to force model onto the GPU.
        device_map={"": 0},
        trust_remote_code=True
    )
    tokenizer = AutoTokenizer.from_pretrained(GPU_BASE_MODEL_ID, trust_remote_code=True)
else:
    # ... (CPU model loading code) ...
    print(f"CPU mode: Loading {CPU_BASE_MODEL_ID}...")
    model = AutoModelForCausalLM.from_pretrained(CPU_BASE_MODEL_ID)
    tokenizer = AutoTokenizer.from_pretrained(CPU_BASE_MODEL_ID)

tokenizer.pad_token = tokenizer.eos_token
model.config.use_cache = False



Loading base model and tokenizer...
GPU mode: Loading meta-llama/Llama-2-7b-hf with 4-bit quantization...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [13]:
# 5. PREPARE AND TOKENIZE THE DATA
print("Formatting and tokenizing the dataset...")
def format_for_finetuning(example):
    return {
        "text": f"### Human:\n{example['generated_instruction']}\n\n### Assistant:\n{example['original_response']}"
    }

def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=512)

formatted_dataset = curated_dataset.map(format_for_finetuning)
tokenized_dataset = formatted_dataset.map(tokenize_function, batched=True)
final_train_dataset = tokenized_dataset.remove_columns(curated_dataset.column_names)

# 6. CONFIGURE LORA (for GPU) AND TRAINER
if IS_GPU_ENV:
    print("GPU mode: Preparing model for LoRA training...")
    model = prepare_model_for_kbit_training(model)
    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        target_modules=["q_proj", "v_proj"],
        bias="none",
        task_type="CAUSAL_LM"
    )
    model = get_peft_model(model, lora_config)

print("Setting up Training Arguments...")
training_args = TrainingArguments(
    output_dir="final_model_checkpoints",
    num_train_epochs=50, # Increased epochs
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    # ## FIX 2 ##: Changed logging steps to see progress more often.
    logging_steps=10,
    report_to="none",
    fp16=True if IS_GPU_ENV else False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=final_train_dataset,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
)

# 7. TRAIN THE FINAL MODEL
print("Starting final model fine-tuning...")
trainer.train()
print("Fine-tuning complete!")

# 8. PRINT 5 EXAMPLE RESPONSES
print("\n--- Generating 5 Example Responses ---")
example_prompts = [
    "What is the most populous country in the world?",
    "Write a short, happy poem about a cat.",
    "Explain the difference between a list and a tuple in Python.",
    "What are three tips for staying healthy?",
    "Who wrote the play 'Hamlet'?"
]

for prompt_text in example_prompts:
    prompt = f"### Human:\n{prompt_text}\n\n### Assistant:\n"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    output_tokens = model.generate(**inputs,
                                     max_new_tokens=100,
                                     do_sample=True,
                                     temperature=0.7,
                                     top_p=0.9)
    response = tokenizer.decode(output_tokens[0], skip_special_tokens=True)

    try:
        response_text = response.split("### Assistant:\n")[1].strip()
    except IndexError:
        response_text = "Failed to generate a valid response."

    print(f"\n----- Prompt: {prompt_text} -----")
    print(response_text)
    print("=" * 30)

# 9. PUSH THE FINAL MODEL TO THE HUB
print(f"\nPushing the final model to the Hub at: {FINAL_MODEL_REPO_ID}")

trainer.model.push_to_hub(FINAL_MODEL_REPO_ID, commit_message="Pushing final LoRA model")
tokenizer.push_to_hub(FINAL_MODEL_REPO_ID, commit_message="Pushing tokenizer")

print("Final model and tokenizer successfully pushed!")
print(f"Model URL: https://huggingface.co/{FINAL_MODEL_REPO_ID}")

Formatting and tokenizing the dataset...


Map:   0%|          | 0/93 [00:00<?, ? examples/s]

Map:   0%|          | 0/93 [00:00<?, ? examples/s]

GPU mode: Preparing model for LoRA training...
Setting up Training Arguments...
Starting final model fine-tuning...


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:929: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...p48nmyj0s/tokenizer.model: 100%|##########|  500kB /  500kB            

No files have been modified since last commit. Skipping to prevent empty commit.


Final model and tokenizer successfully pushed!
Model URL: https://huggingface.co/dnerkar/llama-2-7b-self-aligned-model
